# Extracción de texto desde PDF con PyMuPDF

Este cuaderno implementa el algoritmo de extracción de texto usando la librería **PyMuPDF**.

El objetivo es extraer frases desde documentos PDF digitales, separando cada frase por punto.  
No se realiza OCR porque el estudio se limita a documentos PDF no escaneados.

El algoritmo genera dos archivos de salida:

1. `frases_extraidas_pymupdf.csv`: contiene las frases extraídas.
2. `metricas_pymupdf.csv`: contiene las métricas de rendimiento y calidad básica de la extracción.

Estos archivos permitirán comparar PyMuPDF con otras librerías de extracción que se implementarán posteriormente.

In [ ]:
!pip install pymupdf

In [ ]:
import fitz  # PyMuPDF
import pandas as pd
import re
import time
import os
from google.colab import files

## 1. Carga del archivo PDF

Seleccione el archivo PDF desde su equipo local.  
El archivo debe ser un PDF digital, no escaneado.

In [ ]:
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
nombre_pdf = os.path.basename(pdf_path)

print("Archivo cargado:", nombre_pdf)

## 2. Funciones auxiliares

Estas funciones permiten limpiar el texto, segmentarlo por punto y evaluar si una frase es válida para el corpus inicial.

In [ ]:
def limpiar_texto(texto):
    """
    Limpia saltos de línea, espacios múltiples y caracteres innecesarios.
    """
    texto = texto.replace("\n", " ")
    texto = re.sub(r"\s+", " ", texto)
    texto = texto.strip()
    return texto


def separar_frases_por_punto(texto):
    """
    Separa el texto en frases usando el punto como delimitador.
    La frase resultante conserva el punto final.
    """
    texto = limpiar_texto(texto)
    partes = texto.split(".")

    frases = []
    for parte in partes:
        frase = parte.strip()
        if frase:
            frases.append(frase + ".")

    return frases


def contar_palabras(texto):
    """
    Cuenta palabras considerando secuencias alfabéticas.
    Incluye caracteres propios del español y posibles grafías usadas en Shuar.
    """
    palabras = re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+", texto)
    return len(palabras)


def es_frase_valida(frase, minimo_palabras=3, minimo_caracteres=10):
    """
    Determina si una frase tiene valor mínimo para el corpus.

    Criterios:
    - Debe tener al menos minimo_palabras.
    - Debe tener al menos minimo_caracteres.
    - No debe estar compuesta solo por números, signos o espacios.
    """
    frase_limpia = frase.strip()

    if len(frase_limpia) < minimo_caracteres:
        return False

    if contar_palabras(frase_limpia) < minimo_palabras:
        return False

    if re.fullmatch(r"[\d\W_]+", frase_limpia):
        return False

    return True

## 3. Extracción de frases con PyMuPDF

El algoritmo procesa cada página del PDF, extrae el texto y separa las frases por punto.  
Además, registra errores si alguna página no puede ser procesada.

In [ ]:
def extraer_frases_pymupdf(pdf_path):
    """
    Extrae frases desde un PDF usando PyMuPDF.

    Retorna:
    - DataFrame con frases extraídas.
    - Diccionario con información general del proceso.
    """

    registros = []
    paginas_con_error = 0
    errores = []

    inicio = time.time()

    documento = fitz.open(pdf_path)
    total_paginas = len(documento)

    for numero_pagina, pagina in enumerate(documento, start=1):
        try:
            texto_pagina = pagina.get_text("text")
            frases_pagina = separar_frases_por_punto(texto_pagina)

            for frase in frases_pagina:
                num_palabras = contar_palabras(frase)
                num_caracteres = len(frase)

                registros.append({
                    "id": len(registros) + 1,
                    "pagina": numero_pagina,
                    "frase": frase,
                    "num_palabras": num_palabras,
                    "num_caracteres": num_caracteres,
                    "es_valida": es_frase_valida(frase)
                })

        except Exception as e:
            paginas_con_error += 1
            errores.append({
                "pagina": numero_pagina,
                "error": str(e)
            })

    documento.close()

    fin = time.time()
    tiempo_segundos = round(fin - inicio, 4)

    df_frases = pd.DataFrame(registros)

    info_proceso = {
        "total_paginas": total_paginas,
        "tiempo_segundos": tiempo_segundos,
        "paginas_con_error": paginas_con_error,
        "errores": errores
    }

    return df_frases, info_proceso


df_frases, info_proceso = extraer_frases_pymupdf(pdf_path)

print("Extracción finalizada")
print("Total de páginas:", info_proceso["total_paginas"])
print("Tiempo de extracción:", info_proceso["tiempo_segundos"], "segundos")
print("Total de frases extraídas:", len(df_frases))

df_frases.head(20)

## 4. Cálculo de métricas

Se calculan métricas de cantidad, calidad básica, duplicación, cobertura y eficiencia.

In [ ]:
def calcular_metricas(df_frases, info_proceso, nombre_pdf, libreria="PyMuPDF"):
    """
    Calcula métricas estandarizadas para comparar algoritmos de extracción.
    """

    if df_frases.empty:
        total_caracteres = 0
        total_palabras = 0
        total_frases = 0
        frases_validas = 0
        frases_invalidas = 0
        frases_duplicadas = 0
        porcentaje_frases_validas = 0
        porcentaje_duplicados = 0
        promedio_palabras_por_frase = 0
        promedio_caracteres_por_pagina = 0
    else:
        total_caracteres = int(df_frases["num_caracteres"].sum())
        total_palabras = int(df_frases["num_palabras"].sum())
        total_frases = int(len(df_frases))

        frases_validas = int(df_frases["es_valida"].sum())
        frases_invalidas = int(total_frases - frases_validas)

        frases_duplicadas = int(df_frases.duplicated(subset=["frase"]).sum())

        porcentaje_frases_validas = round((frases_validas / total_frases) * 100, 2) if total_frases > 0 else 0
        porcentaje_duplicados = round((frases_duplicadas / total_frases) * 100, 2) if total_frases > 0 else 0

        promedio_palabras_por_frase = round(df_frases["num_palabras"].mean(), 2)
        promedio_caracteres_por_pagina = round(
            total_caracteres / info_proceso["total_paginas"], 2
        ) if info_proceso["total_paginas"] > 0 else 0

    metricas = {
        "libreria": libreria,
        "archivo_pdf": nombre_pdf,
        "total_paginas": info_proceso["total_paginas"],
        "tiempo_segundos": info_proceso["tiempo_segundos"],
        "total_caracteres": total_caracteres,
        "total_palabras": total_palabras,
        "total_frases": total_frases,
        "frases_validas": frases_validas,
        "frases_invalidas": frases_invalidas,
        "porcentaje_frases_validas": porcentaje_frases_validas,
        "frases_duplicadas": frases_duplicadas,
        "porcentaje_duplicados": porcentaje_duplicados,
        "promedio_palabras_por_frase": promedio_palabras_por_frase,
        "promedio_caracteres_por_pagina": promedio_caracteres_por_pagina,
        "paginas_con_error": info_proceso["paginas_con_error"]
    }

    return pd.DataFrame([metricas])


df_metricas = calcular_metricas(df_frases, info_proceso, nombre_pdf)

df_metricas

## 5. Revisión rápida de frases válidas e inválidas

Esta sección permite observar algunos ejemplos para verificar la calidad básica de la extracción.

In [ ]:
print("Ejemplos de frases válidas:")
display(df_frases[df_frases["es_valida"] == True].head(10))

print("Ejemplos de frases inválidas:")
display(df_frases[df_frases["es_valida"] == False].head(10))

## 6. Guardar los archivos de salida

Se generan dos archivos CSV:

- `frases_extraidas_pymupdf.csv`
- `metricas_pymupdf.csv`

In [ ]:
archivo_frases = "frases_extraidas_pymupdf.csv"
archivo_metricas = "metricas_pymupdf.csv"

df_frases.to_csv(archivo_frases, index=False, encoding="utf-8-sig")
df_metricas.to_csv(archivo_metricas, index=False, encoding="utf-8-sig")

print("Archivo de frases generado:", archivo_frases)
print("Archivo de métricas generado:", archivo_metricas)

In [ ]:
files.download(archivo_frases)
files.download(archivo_metricas)

## 7. Interpretación inicial de los resultados

Para el artículo, las métricas principales que se podrán comparar entre librerías son:

- Total de frases extraídas.
- Porcentaje de frases válidas.
- Porcentaje de frases duplicadas.
- Tiempo de extracción.
- Promedio de palabras por frase.
- Páginas con error.

Cuando se implementen otras librerías, se recomienda mantener exactamente las mismas columnas de salida para facilitar la comparación estadística y la construcción de tablas de resultados.